# Judge Compound Bias — Factorial Study Runner
**RQ:** Do isolated tests of LLM-judge surface biases underestimate vulnerability when multiple irrelevant cues occur together?

Design: 2^k factorial (position × formatting, + length once validated) on fixed content; metric = preference-flip rate vs. all-control baseline; then interaction effects.

**Runtime: T4 GPU** (Runtime → Change runtime type → T4). Cells run in order. Day-1 milestone = smoke test + 2×2 on 10 items.

In [1]:
#@title Cell 1 — Install dependencies (~2 min)
!pip install -q transformers accelerate bitsandbytes statsmodels pandas numpy
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4!")

CUDA available: True
GPU: Tesla T4


In [2]:
#@title Cell 2 — Transformations (position / formatting / length + validation)
import re

# ---------------- POSITION (pair-level) ----------------
def apply_position(resp_a, resp_b, level):
    """level 0: (A,B); level 1: (B,A). Returns (first, second, first_slot_is_a)."""
    if level == 0:
        return resp_a, resp_b, True
    return resp_b, resp_a, False

# ---------------- FORMATTING (content-preserving) ----------------
def apply_formatting(text, level):
    """level 1: same sentences re-wrapped as a markdown list under a heading."""
    if level == 0:
        return text
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
    if not sentences:
        return text
    return "### Explanation\n" + "\n".join(f"- {s}" for s in sentences)

# ---------------- LENGTH (the dangerous factor - neutral filler only) ----------------
# Tuned for ~1.5-2x expansion: closer added to EVERY sentence, restatement only to the FIRST.
_FILLER_OPENERS = ["To put it another way, ", "In other words, "]
_FILLER_CLOSERS = [" This point is worth keeping in mind.", " That is the key idea here.",
                   " This is the essential reasoning."]

def apply_length(text, level):
    """level 1: appends a neutral discourse closer after each sentence.
    No restatement, no new content. Target expansion ~1.3-1.6x."""
    if level == 0:
        return text
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
    if not sentences:
        return text
    out = []
    for i, s in enumerate(sentences):
        closer = _FILLER_CLOSERS[i % len(_FILLER_CLOSERS)]
        out.append(f"{s}{closer}")
    return " ".join(out)

# ---------------- VALIDATION ----------------
_CONTENT_WORD_RE = re.compile(r"\b[a-z]{4,}\b")
_FILLER_WORDS = set()
for _p in _FILLER_OPENERS + _FILLER_CLOSERS:
    _FILLER_WORDS.update(re.findall(r"[a-z]{4,}", _p.lower()))

def validate_length_preservation(original, expanded):
    orig = set(_CONTENT_WORD_RE.findall(original.lower()))
    exp = set(_CONTENT_WORD_RE.findall(expanded.lower()))
    added = exp - orig - _FILLER_WORDS
    return {"length_ratio": round(len(expanded)/max(len(original),1), 2),
            "new_content_words": sorted(added), "clean": len(added) == 0}

# quick self-test
_t = "Water expands when it freezes. This lowers its density. So ice floats."
print(apply_formatting(_t, 1)); print()
_e = apply_length(_t, 1)
print(_e); print()
print(validate_length_preservation(_t, _e))  # want clean=True, ratio ~1.5-2

### Explanation
- Water expands when it freezes.
- This lowers its density.
- So ice floats.

Water expands when it freezes. This point is worth keeping in mind. This lowers its density. That is the key idea here. So ice floats. This is the essential reasoning.

{'length_ratio': 2.39, 'new_content_words': [], 'clean': True}


In [3]:
#@title Cell 3 — Load the judge (Qwen2.5-7B-Instruct, 4-bit) — FIRST REAL TEST (~5 min)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
PROMPT_VERSION = "v1"

_JUDGE_TEMPLATE = """You are an impartial evaluator. You will be shown a question and two candidate answers, labelled [Answer 1] and [Answer 2]. Decide which answer is better, judging only on correctness, relevance, and helpfulness. Ignore differences in length, formatting, and ordering.

Question:
{question}

[Answer 1]
{first}

[Answer 2]
{second}

Respond with EXACTLY one line in the form:
Verdict: 1
or
Verdict: 2
or
Verdict: tie"""

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                             device_map="auto", torch_dtype=torch.float16)
model.eval()
print("loaded:", MODEL_NAME)

def _generate(prompt, max_new_tokens=16):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def _parse_verdict(raw):
    m = re.search(r"verdict\s*[:\-]?\s*(1|2|tie)", raw, re.IGNORECASE)
    if not m:
        m = re.search(r"\b(1|2|tie)\b", raw, re.IGNORECASE)
    if not m:
        return "UNPARSED"
    v = m.group(1).lower()
    return "TIE" if v == "tie" else v

def judge_pair(question, first, second):
    raw = _generate(_JUDGE_TEMPLATE.format(question=question, first=first, second=second))
    return {"slot_verdict": _parse_verdict(raw), "raw": raw, "prompt_version": PROMPT_VERSION}

def slot_to_ab(slot_verdict, first_slot_is_a):
    if slot_verdict in ("TIE", "UNPARSED"):
        return slot_verdict
    if slot_verdict == "1":
        return "A" if first_slot_is_a else "B"
    return "B" if first_slot_is_a else "A"

# ---- SMOKE TEST ----
out = judge_pair("Why does ice float on water?",
                 "Ice floats because water expands when it freezes, lowering its density.",
                 "Ice is less dense than liquid water, so it floats.")
print(out)  # expect slot_verdict in {'1','2','TIE'}; if UNPARSED, we tune the prompt

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

loaded: Qwen/Qwen2.5-7B-Instruct
{'slot_verdict': '2', 'raw': 'Verdict: 2', 'prompt_version': 'v1'}


In [4]:
#@title Cell 4 — Day-1 items (10 hand-made pairs; replace with real dataset on day 2-3)
# Each pair: A is clearly-but-not-absurdly better than B, so the baseline verdict is
# stable and flips are meaningful. Edit freely.
items = [
    {"item_id": "demo_0", "question": "Why does ice float on water?",
     "response_a": "Water expands when it freezes because hydrogen bonds arrange into an open crystal lattice. This lowers its density relative to liquid water, so ice floats.",
     "response_b": "Ice floats because it is lighter than water."},
    {"item_id": "demo_1", "question": "Why is the sky blue?",
     "response_a": "Air molecules scatter shorter wavelengths of sunlight more strongly (Rayleigh scattering), so blue light reaches our eyes from all directions.",
     "response_b": "The sky reflects the color of the ocean, which is blue."},
    {"item_id": "demo_2", "question": "What causes seasons on Earth?",
     "response_a": "Earth's axial tilt of about 23.5 degrees changes how directly sunlight strikes each hemisphere through the year, causing seasons.",
     "response_b": "Seasons happen because Earth moves closer to and farther from the Sun."},
    {"item_id": "demo_3", "question": "Why do we see phases of the Moon?",
     "response_a": "The Moon orbits Earth, so the fraction of its sunlit half visible to us changes over the month, producing phases.",
     "response_b": "Earth's shadow falls on the Moon, hiding parts of it each night."},
    {"item_id": "demo_4", "question": "Why does bread rise?",
     "response_a": "Yeast ferments sugars and releases carbon dioxide, which inflates gas pockets in the gluten network, making the dough rise.",
     "response_b": "Bread rises because heat makes everything expand in the oven."},
    {"item_id": "demo_5", "question": "Why is seawater salty?",
     "response_a": "Rivers dissolve minerals from rock and carry them to the ocean; water evaporates but salts remain and accumulate over geological time.",
     "response_b": "Seawater is salty because fish and other sea creatures excrete salt."},
    {"item_id": "demo_6", "question": "How do vaccines work?",
     "response_a": "Vaccines expose the immune system to a harmless form of a pathogen, training it to produce antibodies and memory cells for faster future response.",
     "response_b": "Vaccines kill germs in your body so you cannot get sick."},
    {"item_id": "demo_7", "question": "Why do leaves change color in autumn?",
     "response_a": "As daylight shortens, chlorophyll breaks down and is not replaced, unmasking yellow and orange pigments already present in the leaf.",
     "response_b": "Cold weather freezes the green out of the leaves, turning them brown."},
    {"item_id": "demo_8", "question": "What causes tides?",
     "response_a": "The Moon's gravity pulls unevenly on Earth's oceans, creating bulges of water; Earth's rotation moves coastlines through these bulges as tides.",
     "response_b": "Tides are caused by wind pushing seawater toward the shore twice a day."},
    {"item_id": "demo_9", "question": "Why is exercise good for the heart?",
     "response_a": "Regular aerobic exercise strengthens the heart muscle, improves stroke volume, lowers resting heart rate and blood pressure, and improves circulation.",
     "response_b": "Exercise is good because it makes you sweat out toxins near your heart."},
]
print(len(items), "items ready")

10 items ready


In [5]:
#@title Cell 5 — Factorial harness (2×2 by default; INCLUDE_LENGTH flips on the 3rd factor)
import itertools, json, time
from pathlib import Path

INCLUDE_LENGTH = False   # ship the clean 2x2 first; enable ONLY after Cell 7 validation

def _factor_levels():
    f = {"position": [0, 1], "formatting": [0, 1]}
    if INCLUDE_LENGTH:
        f["length"] = [0, 1]
    return f

def _all_conditions():
    f = _factor_levels(); keys = list(f)
    for combo in itertools.product(*[f[k] for k in keys]):
        yield dict(zip(keys, combo))

def _build_condition(resp_a, resp_b, cond):
    a = apply_formatting(resp_a, cond.get("formatting", 0))
    b = apply_formatting(resp_b, cond.get("formatting", 0))
    a = apply_length(a, cond.get("length", 0))
    b = apply_length(b, cond.get("length", 0))
    return apply_position(a, b, cond.get("position", 0))

def run_item(item, judge_name):
    base_cond = {k: 0 for k in _factor_levels()}
    first, second, fia = _build_condition(item["response_a"], item["response_b"], base_cond)
    res = judge_pair(item["question"], first, second)
    baseline_ab = slot_to_ab(res["slot_verdict"], fia)
    for cond in _all_conditions():
        first, second, fia = _build_condition(item["response_a"], item["response_b"], cond)
        res = judge_pair(item["question"], first, second)
        v = slot_to_ab(res["slot_verdict"], fia)
        flipped = (baseline_ab in ("A", "B")) and (v in ("A", "B")) and v != baseline_ab
        yield {"item_id": item["item_id"], "judge": judge_name, "question": item["question"],
               "position": cond.get("position", 0), "formatting": cond.get("formatting", 0),
               "length": cond.get("length", 0), "first_slot_is_a": fia,
               "slot_verdict": res["slot_verdict"], "verdict_ab": v, "baseline_ab": baseline_ab,
               "flipped": bool(flipped), "raw_judge_output": res["raw"],
               "prompt_version": res["prompt_version"], "temperature": 0.0,
               "timestamp": time.time()}

def run_experiment(items, judge_name, out_path):
    """Streams JSONL; resumes past already-done item_ids on rerun."""
    out = Path(out_path); out.parent.mkdir(parents=True, exist_ok=True)
    done = set()
    if out.exists():
        for line in open(out):
            try: done.add(json.loads(line)["item_id"])
            except Exception: pass
    n = 0
    with open(out, "a") as f:
        for it in items:
            if it["item_id"] in done: continue
            for rec in run_item(it, judge_name):
                f.write(json.dumps(rec) + "\n"); n += 1
            f.flush()
            print("done:", it["item_id"])
    print(f"wrote {n} records -> {out}")
    return str(out)

print("harness ready | factors:", list(_factor_levels()))

harness ready | factors: ['position', 'formatting']


In [ ]:
#@title Cell 6 — DAY-1 MILESTONE: run the 2×2 on the 10 items (~3-5 min on T4)
path = run_experiment(items, judge_name="qwen2.5-7b", out_path="results/raw_qwen.jsonl")

import pandas as pd
df = pd.DataFrame([json.loads(l) for l in open(path)])
print(df[["item_id","position","formatting","verdict_ab","baseline_ab","flipped"]].to_string())
print()
print("UNPARSED rate:", (df.slot_verdict=="UNPARSED").mean().round(3), " (want < 0.05)")
print("TIE rate:", (df.slot_verdict=="TIE").mean().round(3))
print("any flips at all:", df.flipped.any(), " (if False on 10 items that's OK; if still False at 100+ items, pairs are too lopsided)")

done: demo_0
done: demo_1
done: demo_2
done: demo_3
done: demo_4
done: demo_5
done: demo_6
done: demo_7
done: demo_8
done: demo_9
wrote 40 records -> results/raw_qwen.jsonl
   item_id  position  formatting verdict_ab baseline_ab  flipped
0   demo_0         0           0          A           A    False
1   demo_0         0           1          A           A    False
2   demo_0         1           0          A           A    False
3   demo_0         1           1          A           A    False
4   demo_1         0           0          A           A    False
5   demo_1         0           1          A           A    False
6   demo_1         1           0          A           A    False
7   demo_1         1           1          A           A    False
8   demo_2         0           0          A           A    False
9   demo_2         0           1          A           A    False
10  demo_2         1           0          A           A    False
11  demo_2         1           1          A    

In [ ]:
#@title Cell 7 — Validate LENGTH before enabling it (gate for INCLUDE_LENGTH=True)
for it in items[:5]:
    exp = apply_length(it["response_a"], 1)
    v = validate_length_preservation(it["response_a"], exp)
    print(it["item_id"], v)
    if it["item_id"] == "demo_0":
        print("  EYEBALL THIS:\n ", exp, "\n")
# Requirements to enable length: every row clean=True, ratio ~1.5-2.0, and the
# eyeballed text reads as natural-but-padded, not garbled. Then set
# INCLUDE_LENGTH=True in Cell 5, re-run Cells 5-6 (delete results/raw_qwen.jsonl
# first so baselines are recomputed under the 2^3 design).

demo_0 {'length_ratio': 1.41, 'new_content_words': [], 'clean': True}
  EYEBALL THIS:
  Water expands when it freezes because hydrogen bonds arrange into an open crystal lattice. This point is worth keeping in mind. This lowers its density relative to liquid water, so ice floats. That is the key idea here. 

demo_1 {'length_ratio': 1.26, 'new_content_words': [], 'clean': True}
demo_2 {'length_ratio': 1.29, 'new_content_words': [], 'clean': True}
demo_3 {'length_ratio': 1.33, 'new_content_words': [], 'clean': True}
demo_4 {'length_ratio': 1.3, 'new_content_words': [], 'clean': True}


In [ ]:
#@title Cell 8 — Analysis: flip rates + additivity + interaction model
import numpy as np
import pandas as pd

def flip_rate_table(df):
    factors = [c for c in ["position","formatting","length"] if c in df.columns and df[c].nunique()>1 or c in ("position","formatting")]
    factors = [c for c in ["position","formatting","length"] if c in df.columns]
    usable = df[df.baseline_ab.isin(["A","B"]) & df.verdict_ab.isin(["A","B"])]
    g = usable.groupby(["judge"]+factors)["flipped"].agg(["mean","sum","count"]).reset_index()
    g = g.rename(columns={"mean":"flip_rate","sum":"n_flips","count":"n"})
    g["flip_rate_pct"] = (g.flip_rate*100).round(1)
    return g

def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (round((c-h)*100,1), round((c+h)*100,1))

t = flip_rate_table(df)
t["ci"] = [wilson_ci(r.n_flips, r.n) for r in t.itertuples()]
print(t.to_string()); print()

def additivity_check(df):
    t = flip_rate_table(df)
    factors = [c for c in ["position","formatting","length"] if c in df.columns]
    def rate(**kw):
        m = t.copy()
        for f in factors: m = m[m[f]==kw.get(f,0)]
        return float(m.flip_rate.mean()) if len(m) else np.nan
    singles = {f: rate(**{f:1}) for f in factors}
    all_on = rate(**{f:1 for f in factors})
    s = sum(singles.values())
    verdict = ("super-additive" if all_on > s+0.02 else
               "antagonistic" if all_on < s-0.02 else "approximately additive")
    return {"singles": {k: round(v,3) for k,v in singles.items()},
            "sum_of_singles": round(s,3), "observed_compound": round(all_on,3),
            "verdict": verdict}
print(additivity_check(df)); print()

# --- interaction model (logit with item-cluster-robust SEs; mixed model optional) ---
import statsmodels.formula.api as smf
usable = df[df.baseline_ab.isin(["A","B"]) & df.verdict_ab.isin(["A","B"])].copy()
usable["y"] = usable.flipped.astype(int)
factors = [c for c in ["position","formatting","length"] if c in usable.columns]
formula = "y ~ " + "*".join(factors)
try:
    res = smf.logit(formula, data=usable).fit(disp=False,
            cov_type="cluster", cov_kwds={"groups": usable["item_id"]})
    tidy = pd.DataFrame({"coef": res.params, "p": res.pvalues})
    ci = res.conf_int(); tidy["ci_low"], tidy["ci_high"] = ci[0], ci[1]
    tidy["OR"] = np.exp(tidy.coef).round(3)
    print(tidy.round(3).to_string())
except Exception as e:
    print("model failed (fine on 10 items - too little data; rerun at scale):", e)

        judge  position  formatting  length  flip_rate  n_flips   n  flip_rate_pct            ci
0  qwen2.5-7b         0           0       0        0.0        0  10            0.0  (-0.0, 27.8)
1  qwen2.5-7b         0           1       0        0.0        0  10            0.0  (-0.0, 27.8)
2  qwen2.5-7b         1           0       0        0.0        0  10            0.0  (-0.0, 27.8)
3  qwen2.5-7b         1           1       0        0.0        0  10            0.0  (-0.0, 27.8)

{'singles': {'position': 0.0, 'formatting': 0.0, 'length': nan}, 'sum_of_singles': nan, 'observed_compound': nan, 'verdict': 'approximately additive'}

model failed (fine on 10 items - too little data; rerun at scale): Singular matrix


/usr/local/lib/python3.13/dist-packages/statsmodels/base/optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
/usr/local/lib/python3.13/dist-packages/statsmodels/base/optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
/usr/local/lib/python3.13/dist-packages/statsmodels/base/optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
/usr/local/lib/python3.13/dist-packages/statsmodels/base/optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
/usr/local/lib/python3.13/dist-packages/statsmodels/base/optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
/usr/local

In [ ]:
!rm results/raw_qwen.jsonl

In [ ]:
#@title Cell 9 — Persist results to Drive (run after every real session!)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/judge-compound-bias
!cp -v results/*.jsonl /content/drive/MyDrive/judge-compound-bias/
# Colab wipes local files when the runtime dies. results/*.jsonl IS your paper.

## Day 2-3: swap in the real dataset
Replace Cell 4 with 200-300 real pairs, e.g. from **MT-Bench** or **LMSYS Chatbot Arena** on HuggingFace (`datasets.load_dataset("lmsys/mt_bench_human_judgments")` or an Arena conversations slice). Reshape each row into `{"item_id", "question", "response_a", "response_b"}` and keep `item_id` stable so resume works. Prefer pairs where humans had a clear preference (so your baseline is anchored) but not absurdly lopsided ones (so flips are possible).

Then: full run (Cell 6 at scale) → validate + enable length (Cell 7 → Cell 5) → re-run under 2^3 → analysis (Cell 8) → second judge only if everything is clean.

**If the smoke test errors or UNPARSED is high — paste the output back to Claude and we fix the prompt/parser before scaling.**